In [1]:
%run_nb spark-start --data-format iceberg

Args: Namespace(data_format='iceberg', port_offset=2) - unknown_args: []
Spark version: 4.1.2, Driver memory: 16g, Executor memory: 8g, Service: jupyter-spark-4.1, Data format: iceberg
Spark packages: org.apache.iceberg:iceberg-spark-runtime-4.1_2.13:1.11.0
Spark extensions: org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions
Spark catalog configs: {'spark.sql.catalog.local': 'org.apache.iceberg.spark.SparkCatalog', 'spark.sql.catalog.local.type': 'hadoop', 'spark.sql.catalog.local.warehouse': 'file:///home/jovyan/work/data/datalake/jupyter-spark-4.1/spark-4.1/iceberg/warehouse'}


,catalog
0,spark_catalog


Version,4.1.2
Master,local[2]
AppName,main


               total        used        free      shared  buff/cache   available
Mem:            62Gi        17Gi       3.4Gi       437Mi        42Gi        45Gi
Swap:          8.0Gi          0B       8.0Gi


In [2]:
from pyspark.sql import Row

data = [
    Row(id=1, name="Alice"),
    Row(id=2, name="Bob"),
    Row(id=3, name="Charlie"),
]

df = spark.createDataFrame(data)

In [3]:
# Create an Iceberg namespace/database
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.demo")

DataFrame[]

In [4]:
# Create the Iceberg table and write the DataFrame
(
    df.writeTo("local.demo.people")
      .using("iceberg")
      .createOrReplace()
)

In [5]:
spark.table("local.demo.people").show()

+---+-------+
| id|   name|
+---+-------+
|  1|  Alice|
|  2|    Bob|
|  3|Charlie|
+---+-------+



In [6]:
new_data = spark.createDataFrame([
    (4, "David"),
    (5, "Emma"),
], ["id", "name"])

new_data.writeTo("local.demo.people").append()

In [7]:
viewdf(new_data)

,id,name
0,4,David
1,5,Emma


In [8]:
%sql SELECT * FROM local.demo.people LIMIT 20

,id,name
0,4,David
1,5,Emma
2,1,Alice
3,2,Bob
4,3,Charlie


In [9]:
%sql SHOW NAMESPACES IN local

,namespace
0,demo
1,fifa


In [10]:
%sql SHOW CATALOGS;
%sql SHOW TABLES;

,catalog
0,local
1,spark_catalog


,namespace,tableName,isTemporary


In [16]:
%%sql
SELECT COUNT(*) AS data_file_count
FROM local.demo.people.data_files;

SELECT
    file_path,
    file_format,
    record_count,
    file_size_in_bytes
FROM local.demo.people.data_files;

-- Show table definition, including partitioning
DESCRIBE TABLE EXTENDED local.demo.people;

-- To inspect file-level statistics used for data skipping
SELECT
    file_path,
    record_count,
    lower_bounds,
    upper_bounds,
    null_value_counts
FROM local.demo.people.files;

-- To see the partition layout:
SELECT *
FROM local.demo.people.partitions;

-- To see which sort-order ID each data file uses:
SELECT
    file_path,
    sort_order_id
FROM local.demo.people.files;


,data_file_count
0,4


,file_path,file_format,record_count,file_size_in_bytes
0,file:/home/jovyan/work/data/datalake/jupyter-s...,PARQUET,1,686
1,file:/home/jovyan/work/data/datalake/jupyter-s...,PARQUET,1,679
2,file:/home/jovyan/work/data/datalake/jupyter-s...,PARQUET,1,686
3,file:/home/jovyan/work/data/datalake/jupyter-s...,PARQUET,2,689


,col_name,data_type,comment
0,id,bigint,None
1,name,string,None
2,,,
3,# Metadata Columns,,
4,_spec_id,int,
5,_partition,struct<>,
6,_file,string,
7,_pos,bigint,
8,_deleted,boolean,
9,,,


,file_path,record_count,lower_bounds,upper_bounds,null_value_counts
0,file:/home/jovyan/work/data/datalake/jupyter-s...,1,"{1: b'\x04\x00\x00\x00\x00\x00\x00\x00', 2: b'...","{1: b'\x04\x00\x00\x00\x00\x00\x00\x00', 2: b'...","{1: 0, 2: 0}"
1,file:/home/jovyan/work/data/datalake/jupyter-s...,1,"{1: b'\x05\x00\x00\x00\x00\x00\x00\x00', 2: b'...","{1: b'\x05\x00\x00\x00\x00\x00\x00\x00', 2: b'...","{1: 0, 2: 0}"
2,file:/home/jovyan/work/data/datalake/jupyter-s...,1,"{1: b'\x01\x00\x00\x00\x00\x00\x00\x00', 2: b'...","{1: b'\x01\x00\x00\x00\x00\x00\x00\x00', 2: b'...","{1: 0, 2: 0}"
3,file:/home/jovyan/work/data/datalake/jupyter-s...,2,"{1: b'\x02\x00\x00\x00\x00\x00\x00\x00', 2: b'...","{1: b'\x03\x00\x00\x00\x00\x00\x00\x00', 2: b'...","{1: 0, 2: 0}"


,record_count,file_count,total_data_file_size_in_bytes,position_delete_record_count,position_delete_file_count,equality_delete_record_count,equality_delete_file_count,last_updated_at,last_updated_snapshot_id
0,5,4,2740,0,0,0,0,2026-07-22 19:12:07.218,5161294362761053818


,file_path,sort_order_id
0,file:/home/jovyan/work/data/datalake/jupyter-s...,0
1,file:/home/jovyan/work/data/datalake/jupyter-s...,0
2,file:/home/jovyan/work/data/datalake/jupyter-s...,0
3,file:/home/jovyan/work/data/datalake/jupyter-s...,0


In [17]:
%run_nb spark-show

Job Id ▾,Description,Submitted,Duration,Stages: Succeeded/Total,Tasks (for all stages): Succeeded/Total
27,toPandas at /home/jovyan/work/spark/lib.py:82 toPandas at /home/jovyan/work/spark/lib.py:82,2026/07/22 19:16:17,29 ms,1/1,1/1
26,toPandas at /home/jovyan/work/spark/lib.py:82 toPandas at /home/jovyan/work/spark/lib.py:82,2026/07/22 19:16:17,9 ms,1/1,1/1
25,toPandas at /home/jovyan/work/spark/lib.py:82 toPandas at /home/jovyan/work/spark/lib.py:82,2026/07/22 19:16:17,16 ms,1/1,1/1
24,toPandas at /home/jovyan/work/spark/lib.py:82 toPandas at /home/jovyan/work/spark/lib.py:82,2026/07/22 19:16:17,16 ms,1/1,1/1
23,toPandas at /home/jovyan/work/spark/lib.py:82 toPandas at /home/jovyan/work/spark/lib.py:82,2026/07/22 19:16:17,10 ms,1/1 (1 skipped),1/1 (1 skipped)
22,toPandas at /home/jovyan/work/spark/lib.py:82 toPandas at /home/jovyan/work/spark/lib.py:82,2026/07/22 19:16:17,13 ms,1/1,1/1
21,toPandas at /home/jovyan/work/spark/lib.py:82 toPandas at /home/jovyan/work/spark/lib.py:82,2026/07/22 19:15:46,30 ms,1/1,1/1
20,toPandas at /home/jovyan/work/spark/lib.py:82 toPandas at /home/jovyan/work/spark/lib.py:82,2026/07/22 19:15:46,13 ms,1/1,1/1
19,toPandas at /home/jovyan/work/spark/lib.py:82 toPandas at /home/jovyan/work/spark/lib.py:82,2026/07/22 19:15:46,21 ms,1/1,1/1
18,toPandas at /home/jovyan/work/spark/lib.py:82 toPandas at /home/jovyan/work/spark/lib.py:82,2026/07/22 19:15:46,9 ms,1/1 (1 skipped),1/1 (1 skipped)


Stage Id ▾,Description,Submitted,Duration,Tasks: Succeeded/Total,Input,Output,Shuffle Read,Shuffle Write
33,toPandas at /home/jovyan/work/spark/lib.py:82 +details org.apache.spark.sql.classic.Dataset.collectToPython(Dataset.scala:2081) jdk.internal.reflect.GeneratedMethodAccessor103.invoke(Unknown Source) java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:52) java.base/java.lang.reflect.Method.invoke(Method.java:580) py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244) py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374) py4j.Gateway.invoke(Gateway.java:282) py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132) py4j.commands.CallCommand.execute(CallCommand.java:79) py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:184) py4j.ClientServerConnection.run(ClientServerConnection.java:108) java.base/java.lang.Thread.run(Thread.java:1583),2026/07/22 19:16:17,23 ms,1/1,14.0 KiB,,,
32,toPandas at /home/jovyan/work/spark/lib.py:82 +details org.apache.spark.sql.classic.Dataset.collectToPython(Dataset.scala:2081) jdk.internal.reflect.GeneratedMethodAccessor103.invoke(Unknown Source) java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:52) java.base/java.lang.reflect.Method.invoke(Method.java:580) py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244) py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374) py4j.Gateway.invoke(Gateway.java:282) py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132) py4j.commands.CallCommand.execute(CallCommand.java:79) py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:184) py4j.ClientServerConnection.run(ClientServerConnection.java:108) java.base/java.lang.Thread.run(Thread.java:1583),2026/07/22 19:16:17,4 ms,1/1,,,,
31,toPandas at /home/jovyan/work/spark/lib.py:82 +details org.apache.spark.sql.classic.Dataset.collectToPython(Dataset.scala:2081) jdk.internal.reflect.GeneratedMethodAccessor103.invoke(Unknown Source) java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:52) java.base/java.lang.reflect.Method.invoke(Method.java:580) py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244) py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374) py4j.Gateway.invoke(Gateway.java:282) py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132) py4j.commands.CallCommand.execute(CallCommand.java:79) py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:184) py4j.ClientServerConnection.run(ClientServerConnection.java:108) java.base/java.lang.Thread.run(Thread.java:1583),2026/07/22 19:16:17,11 ms,1/1,14.0 KiB,,,
30,toPandas at /home/jovyan/work/spark/lib.py:82 +details org.apache.spark.sql.classic.Dataset.collectToPython(Dataset.scala:2081) jdk.internal.reflect.GeneratedMethodAccessor103.invoke(Unknown Source) java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:52) java.base/java.lang.reflect.Method.invoke(Method.java:580) py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244) py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374) py4j.Gateway.invoke(Gateway.java:282) py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132) py4j.commands.CallCommand.execute(CallCommand.java:79) py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:184) py4j.ClientServerConnection.run(ClientServerConnection.java:108) java.base/java.lang.Thread.run(Thread.java:1583),2026/07/22 19:16:17,12 ms,1/1,14.0 KiB,,,
29,toPandas at /home/jovyan/work/spark/lib.py:82 +details org.apache.spark.sql.execution.SQLExecution$.$anonfun$withThreadLocalCaptured$2(SQLExecution.scala:329) java.base/java.util.concurrent.CompletableFuture$AsyncSupply.run(CompletableFuture.java:1768) java.base/java.util.concurrent.ThreadPoolExecutor.runWorker(ThreadPoolExecutor.java:1144) ja

Version,4.1.2
Master,local[2]
AppName,main


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 59612)
Traceback (most recent call last):
  File "/opt/conda/lib/python3.13/socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
    ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.13/socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.13/socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
    ~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.13/socketserver.py", line 766, in __init__
    self.handle()
    ~~~~~~~~~~~^^
  File "/usr/local/spark/python/pyspark/accumulators.py", line 303, in handle
    poll(accum_updates)
    ~~~~^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/pyspark/accumu